# PACE Phase 11D — authentic AMPGAN v2 checkpoint retrieval

This is an **analysis-only** follow-up to your verified Phase 11C source inventory. It checks the exact official GitLab release `v2.0` (`2102eb5d26dd9a3c12a5a0bad7c7bb3c58b80c70`), downloads **only four LFS checkpoint blobs**, validates SHA-256 and size against the official repository pointers, and packages the generation code for the next step.

**It does not claim to generate peptides or measure activity.** If model retrieval fails, upload its diagnostic ZIP instead of replacing the files. Do not restart the Colab runtime if download succeeds.

In [ ]:
# PHASE 11D: Official AMPGAN v2 checkpoint recovery + source capture.
# This notebook intentionally DOES NOT synthesize peptides or claim a comparison.
import os,sys,json,hashlib,subprocess,shutil,time,zipfile,platform,re
from pathlib import Path

OUT = Path('/content/PACE_Phase11D_Audit')
OUT.mkdir(parents=True,exist_ok=True)
REPO = Path('/content/ampgan_v2_official')
SOURCE_URL = 'https://gitlab.com/vail-uvm/amp-gan.git'
TAG = 'v2.0'
EXPECTED_COMMIT = '2102eb5d26dd9a3c12a5a0bad7c7bb3c58b80c70'
MODEL_PREFIX = 'models/amp_gan_2020-07-08_24/gan_1606/'
REQUIRED = [
 MODEL_PREFIX + 'saved_model.pb',
 MODEL_PREFIX + 'variables/variables.data-00000-of-00002',
 MODEL_PREFIX + 'variables/variables.data-00001-of-00002',
 MODEL_PREFIX + 'variables/variables.index'
]
meta={
 'phase':'11D',
 'scope':'AMPGAN v2 original checkpoint recovery and inference-source verification ONLY',
 'official_origin':SOURCE_URL,'expected_tag':TAG,'expected_commit':EXPECTED_COMMIT,
 'platform':platform.platform(), 'python':sys.version,
 'checkpoint_files':{},'success':False,'generated_sequences_count':0,
 'notes':['No peptides generated or benchmarked by this audit.'],
}

def execute(cmd, cwd=None, env=None, check=True, timeout=None):
 print('Running:', ' '.join(map(str,cmd)),flush=True)
 p=subprocess.run(cmd,cwd=str(cwd) if cwd else None,env=env,text=True,
                  stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
 print((p.stdout or '')[-6500:],flush=True)
 if check and p.returncode!=0:raise RuntimeError('Command failed (%s): %s'%(p.returncode,' '.join(cmd)))
 return p

def sha256(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
 return h.hexdigest()

def lfs_pointer_at_tag(rel):
 raw=execute(['git','show',TAG+':'+rel],cwd=REPO).stdout.strip()
 size=re.search(r'(?m)^size (\d+)\s*$',raw)
 oid=re.search(r'(?m)^oid sha256:([a-f0-9]{64})\s*$',raw)
 if not (size and oid):raise RuntimeError('Missing official Git LFS pointer at tag: '+rel)
 return int(size.group(1)),oid.group(1)

try:
 if not shutil.which('git'): raise RuntimeError('git executable missing')
 chk=execute(['git','lfs','version'],check=False)
 if chk.returncode!=0:
  execute(['apt-get','update','-qq'])
  execute(['apt-get','install','-y','-qq','git-lfs'])
  execute(['git','lfs','install'])
 meta['git_lfs_version']=execute(['git','lfs','version']).stdout.strip()
 if (REPO/'.git').exists():
  print('Reusing existing local git checkout: ',REPO)
 else:
  env=os.environ.copy();env['GIT_LFS_SKIP_SMUDGE']='1'
  execute(['git','clone','--branch',TAG,'--depth','1',SOURCE_URL,str(REPO)],env=env)
  execute(['git','lfs','install','--local'],cwd=REPO)
 commit=execute(['git','rev-parse','HEAD'],cwd=REPO).stdout.strip()
 meta['commit']=commit
 meta['remote']=execute(['git','remote','get-url','origin'],cwd=REPO).stdout.strip()
 assert commit==EXPECTED_COMMIT, 'Release commit mismatch; stopping'
 assert 'vail-uvm/amp-gan' in meta['remote'].lower(), 'Unexpected repository origin'
 # Only four checkpoint files; avoid pulling hundreds of historical training inputs.
 pat=MODEL_PREFIX+'**'
 p=execute(['git','lfs','pull','--include',pat,'--exclude',''],cwd=REPO,check=False)
 meta['lfs_pull_returncode']=p.returncode
 meta['lfs_pull_tail']=(p.stdout or '')[-2500:]
 all_ok=True
 for rel in REQUIRED:
  expected_size,expected_hash=lfs_pointer_at_tag(rel)
  f=REPO/rel
  size=f.stat().st_size if f.exists() else None
  # Exact blob checksum validates that we have real LFS objects and correct versions.
  h=sha256(f) if size is not None else None
  valid=(size==expected_size and h==expected_hash)
  all_ok=all_ok and valid
  meta['checkpoint_files'][rel]={'actual_bytes':size,'expected_bytes':expected_size,
    'sha256_actual':h,'sha256_expected':expected_hash,'valid_lfs_blob':valid}
  print(rel,'VERIFIED' if valid else 'MISSING_OR_POINTER',flush=True)
 # Always export official Python generation source to permit faithful next-step inference.
 src=OUT/'source_files';src.mkdir(exist_ok=True)
 for rel in ['README.md','ampgan/README.md','ampgan/generate_samples.py',
             'ampgan/model.py','ampgan/cli_utils.py','ampgan/data_utils.py',
             'ampgan/train.py','.gitattributes']:
  origin=REPO/rel
  if origin.exists() and origin.stat().st_size<250000:
   f=src/rel;f.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(origin,f)
 meta['source_files_captured']=[str(x.relative_to(src)) for x in src.rglob('*') if x.is_file()]
 meta['success']=bool(all_ok)
 meta['conclusion']=('ORIGINAL MODEL FILES VERIFIED; GENERATION SCRIPT READY TO REVIEW'
     if all_ok else 'INCOMPLETE: one or more original Git LFS checkpoint files not retrieved')
except Exception as e:
 meta['error']=type(e).__name__+': '+str(e)
 meta['conclusion']='BLOCKED: inspect error and official upstream availability'
finally:
 (OUT/'checkpoint_recovery_report.json').write_text(json.dumps(meta,indent=2))
 (OUT/'README.txt').write_text('Authentic AMPGAN v2 checkpoint and source readiness audit.\n'
    'Generated peptides: NONE. No predicted MIC/HC50 comparison conducted.\n'
    'Do not mistake LFS pointer files for model weights.\n'
    'Retain Colab runtime if checkpoint recovery succeeded for the next inference step.\n')
 ZIP = Path('/content/PACE_Phase11D_AMPGANv2_Checkpoint_Readiness.zip')
 with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
  for f in OUT.rglob('*'):
   if f.is_file():z.write(f,f.relative_to(OUT))
 with zipfile.ZipFile(ZIP) as z:assert z.testzip() is None
 print('\nFINAL STATUS:',meta.get('conclusion'),flush=True)
 print('OUTPUT FILE:',ZIP,'BYTES:',ZIP.stat().st_size,flush=True)
 print('IMPORTANT: SavedModel weights excluded from ZIP to avoid huge upload.\n',flush=True)
 try:
  from google.colab import files
  files.download(str(ZIP))
 except ImportError:pass


**Upload** `PACE_Phase11D_AMPGANv2_Checkpoint_Readiness.zip` to ChatGPT. The checkpoint weights are intentionally excluded from the ZIP and remain in the Colab runtime. The generated code files are copied from the verified release for independent inspection. **No manuscript revisions** are made.